In [0]:
import re
import pandas as pd
from pyspark.sql import SparkSession
from dotenv import load_dotenv
import os

load_dotenv()

ENV = os.getenv("ENV", "local")

if ENV == "local":
    base_path = "./data/"
    output_path = "./delta/"
else:
    base_path = "s3://ds-miniproject-datasets/"


spark = (
    SparkSession.builder.appName("ds-miniproject")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)


def to_snake_case(name):
    cleaned = re.sub(r"[^a-zA-Z0-9_]", "", name.strip())
    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", cleaned)
    snake = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1)
    return re.sub(r"_+", "_", snake).lower().strip("_")


def load_excel(path):
    if ENV == "local":
        pdf = pd.read_excel(path, sheet_name=0, header=0)
        df = spark.createDataFrame(pdf.astype(str))
        return df
    else:
        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(path)
        )
        return df


def clean_column_names(df):
    return df.toDF(*[to_snake_case(c) for c in df.columns])


files = {
    "raw_billings": "billings.csv",
    "raw_cc_calls": "cc_calls.csv",
    "raw_emails": "emails.csv",
    "raw_renewal_calls": "renewal_calls.csv",
}

for table_name, filename in files.items():
    print(f"Ingesting {filename}...")

    df = load_excel(base_path + filename)
    df = clean_column_names(df)

    print(f"  Shape: {df.count()} rows x {len(df.columns)} cols")

    if ENV == "local":
        df.write.format("delta").mode("overwrite").save(f"{output_path}{table_name}")
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(f"`workspace`.`ds-raw-datasets`.`{table_name}`")